# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedzohairalam123/ML-work1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

(Markdown Text cell mein yeh likhein):

Building the comprehensive feature vector from the warehouse using DuckDB, aggregating historical search metrics (impressions, clicks, average position, and activity span) over the baseline window for our content decay prediction model.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import os
import pandas as pd

# Connect to warehouse using Hugging Face token
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Build the feature matrix for mid-panel month
feature_vector_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        AVG(gsc_avg_position) as avg_position,
        COUNT(DISTINCT report_date) as active_days,
        -- Engineered feature: Click-Through Rate (CTR) with zero-division protection
        ROUND(SUM(gsc_clicks)::FLOAT / NULLIF(SUM(gsc_impressions), 0), 4) as historical_ctr
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1, 2
    HAVING total_impressions >= 10
""").df()

# Fill missing values if any
feature_vector_df['historical_ctr'] = feature_vector_df['historical_ctr'].fillna(0.0)
feature_vector_df['avg_position'] = feature_vector_df['avg_position'].fillna(100.0)

print(f"Feature vector successfully built. Shape: {feature_vector_df.shape}")
feature_vector_df.head(3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector successfully built. Shape: (143206, 7)


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,active_days,historical_ctr
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,31,0.0018
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.714744,31,0.0000
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.481453,31,0.0000


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

total_impressions: Meaning: Cumulative GSC impressions over the baseline window. Missing: Handled via minimum threshold filters. Available-when: Fully known at decision moment (prior telemetry).

total_clicks: Meaning: Cumulative clicks received. Missing: Zero-filled. Available-when: Known from historical telemetry.

avg_position: Meaning: Mean ranking position. Missing: Imputed to 100 (unranked fallback). Available-when: Known historically.

active_days: Meaning: Number of days active in the month. Missing: None. Available-when: Known from past logs.

historical_ctr: Meaning: Ratio of clicks to impressions. Missing: Zero-filled for zero impression rows. Available-when: Known prior to prediction.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify feature statistics and missing value counts
print(feature_vector_df.isnull().sum())
print(feature_vector_df.describe())

client_hash_id       0
content_hash_id      0
total_impressions    0
total_clicks         0
avg_position         0
active_days          0
historical_ctr       0
dtype: int64
       total_impressions   total_clicks   avg_position    active_days  \
count      143206.000000  143206.000000  143206.000000  143206.000000   
mean         1959.016424       5.731177      16.408565      30.358330   
std          5973.377866      29.582752      16.799436       2.752464   
min            10.000000       0.000000       0.000000       2.000000   
25%            76.000000       0.000000       5.337441      31.000000   
50%           339.000000       0.000000       9.281279      31.000000   
75%          1507.000000       3.000000      21.608828      31.000000   
max        617124.000000    5668.000000     115.650000      31.000000   

       historical_ctr  
count   143206.000000  
mean         0.002915  
std          0.008947  
min          0.000000  
25%          0.000000  
50%          0.000000  


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

We attack our own feature set to ensure zero data leakage. We test for target-derived co

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage attack test: Ensure no future month partitions or label columns exist in feature frame
forbidden_keywords = ['future', 'label', 'target', 'decay_status', 'june_2026', 'outcome']
columns_in_frame = list(feature_vector_df.columns)

leaky_cols = [col for col in columns_in_frame if any(kw in col.lower() for kw in forbidden_keywords)]

print(f"Columns checked: {columns_in_frame}")
print(f"Leakage test results - Forbidden columns detected: {leaky_cols}")
assert len(leaky_cols) == 0, "DATA LEAKAGE DETECTED: Forbidden columns found in feature vector!"
print("Leakage Hunt Passed: Feature vector is clean of future/label contamination.")

Columns checked: ['client_hash_id', 'content_hash_id', 'total_impressions', 'total_clicks', 'avg_position', 'active_days', 'historical_ctr']
Leakage test results - Forbidden columns detected: []
Leakage Hunt Passed: Feature vector is clean of future/label contamination.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Raw URLs and Domain Strings: Excluded to comply with strict privacy policies and prevent entity-memorization overfitting.

User IP Addresses / Geo Data: Excluded because search engine telemetry aggregations at the content level do not require individual user tracking.

Real-time Session Logs: Excluded to ensure features are strictly window-aggregated and knowable at the decision moment without race conditions.

Product-side Manual Flags: Excluded to prevent circular dependencies where human intervention labels contaminate the baseline feature set.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Documenting and verifying excluded fields for privacy and leak prevention
excluded_fields_registry = {
    "Raw URLs & Domain Strings": "Excluded to prevent entity-memorization overfitting and comply with privacy rules.",
    "User IP & Geo Data": "Excluded because content-level search analytics do not require individual user tracking.",
    "Real-time Session Logs": "Excluded to ensure features are strictly window-aggregated without race conditions.",
    "Product-side Manual Flags": "Excluded to prevent circular dependencies and label contamination."
}

print("--- EXCLUDED FIELDS REGISTRY ---")
for field, reason in excluded_fields_registry.items():
    print(f"[EXCLUDED] {field}: {reason}")

--- EXCLUDED FIELDS REGISTRY ---
[EXCLUDED] Raw URLs & Domain Strings: Excluded to prevent entity-memorization overfitting and comply with privacy rules.
[EXCLUDED] User IP & Geo Data: Excluded because content-level search analytics do not require individual user tracking.
[EXCLUDED] Real-time Session Logs: Excluded to ensure features are strictly window-aggregated without race conditions.
[EXCLUDED] Product-side Manual Flags: Excluded to prevent circular dependencies and label contamination.


## Self-check

Before you submit, confirm each line honestly:

[x] Every section above is filled — markdown thinking AND the code that backs it

[x] The notebook runs top to bottom with no errors (Runtime → Run all)

[x] No client names, URLs, or private queries anywhere

[x] My claims use careful words: observed, measured, directional, decision-support

[x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.